# Análise Exploratória – Olist
Este notebook lê os CSVs baixados do Kaggle e gera indicadores e tabelas resumidas. Coloque os arquivos em `../data/raw/` antes de executar.

In [ ]:
from pathlib import Path
import pandas as pd

RAW = Path('../data/raw')


In [ ]:
def ler(nome, datas=None):
    caminho = RAW / nome
    if not caminho.exists():
        raise FileNotFoundError(f'Coloque {nome} na pasta data/raw')
    return pd.read_csv(caminho, parse_dates=datas or [], low_memory=False)

orders = ler('olist_orders_dataset.csv', ['order_purchase_timestamp'])
customers = ler('olist_customers_dataset.csv')
items = ler('olist_order_items_dataset.csv')
payments = ler('olist_order_payments_dataset.csv')
reviews = ler('olist_order_reviews_dataset.csv')
products = ler('olist_products_dataset.csv')


## 1. Conhecimento inicial e qualidade dos dados

In [ ]:
for nome, df in {'pedidos':orders, 'clientes':customers, 'itens':items, 'pagamentos':payments, 'avaliacoes':reviews, 'produtos':products}.items():
    print('\n', nome, df.shape)
    display(df.head(3))
    display(df.isna().sum().sort_values(ascending=False).head(8))


## 2. Limpeza e criação de variáveis

In [ ]:
orders = orders.drop_duplicates(subset='order_id').copy()
customers = customers.drop_duplicates(subset='customer_id').copy()
products = products.drop_duplicates(subset='product_id').copy()
orders['order_purchase_timestamp'] = pd.to_datetime(orders['order_purchase_timestamp'], errors='coerce')
orders['ano_compra'] = orders['order_purchase_timestamp'].dt.year
orders['mes_compra'] = orders['order_purchase_timestamp'].dt.to_period('M').astype('string')
customers['customer_state'] = customers['customer_state'].astype('string').str.strip().str.upper()
items['price'] = pd.to_numeric(items['price'], errors='coerce')
items['freight_value'] = pd.to_numeric(items['freight_value'], errors='coerce')
items['valor_total_item'] = items['price'].fillna(0) + items['freight_value'].fillna(0)


## 3. Indicadores e análises descritivas

In [ ]:
indicadores = pd.DataFrame({
    'indicador':['pedidos','clientes únicos','valor dos produtos','valor do frete','média das avaliações'],
    'resultado':[orders.order_id.nunique(), customers.customer_unique_id.nunique(), items.price.sum(), items.freight_value.sum(), reviews.review_score.mean()]
})
display(indicadores)


In [ ]:
pedidos_mes = orders.dropna(subset=['mes_compra']).groupby('mes_compra').order_id.nunique().reset_index(name='pedidos')
display(pedidos_mes.tail(15))
clientes_estado = customers.groupby('customer_state').customer_unique_id.nunique().sort_values(ascending=False).head(10)
display(clientes_estado)
pagamentos = payments.groupby('payment_type').agg(valor=('payment_value','sum'), transacoes=('order_id','count')).sort_values('valor', ascending=False)
display(pagamentos)
display(reviews.review_score.value_counts().sort_index())


## 4. Interpretação
Compare os resultados com os indicadores do relatório e com os gráficos do Looker Studio. Observe tendências mensais, concentração geográfica, distribuição das notas e preferência de pagamento. Não some valores de tabelas diferentes depois de junções sem verificar a granularidade, pois isso pode duplicar os totais.